# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
import warnings
warnings.filterwarnings("ignore")

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv('data/dayofweek.csv')
df.head(5)

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,-0.756764,-2.562352,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,-0.724861,-2.562352,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,-0.692958,-2.562352,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,-0.661055,-2.562352,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [4]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [5]:
def crossval(n_splits, x, y, model):
    scores = cross_validate(model, x, y, cv=n_splits, return_train_score=True)
    for n in range(n_splits):
        print(f'train - {scores["train_score"][n]:.5f}   |   test - {scores["test_score"][n]:.5f}')
        print(f"Average accuracy on crossval is {scores['test_score'].mean():.5f}")
    print(f"Std is {scores['test_score'].std():.5f}")

In [6]:
%%time

log_model_base = LogisticRegression(random_state=21, fit_intercept=False)
crossval(10, X_train, y_train, log_model_base)

train - 0.62819   |   test - 0.59259
Average accuracy on crossval is 0.60239
train - 0.64716   |   test - 0.62963
Average accuracy on crossval is 0.60239
train - 0.63479   |   test - 0.57037
Average accuracy on crossval is 0.60239
train - 0.65540   |   test - 0.61481
Average accuracy on crossval is 0.60239
train - 0.63314   |   test - 0.57778
Average accuracy on crossval is 0.60239
train - 0.64056   |   test - 0.59259
Average accuracy on crossval is 0.60239
train - 0.64221   |   test - 0.65926
Average accuracy on crossval is 0.60239
train - 0.65952   |   test - 0.56296
Average accuracy on crossval is 0.60239
train - 0.64333   |   test - 0.59701
Average accuracy on crossval is 0.60239
train - 0.63591   |   test - 0.62687
Average accuracy on crossval is 0.60239
Std is 0.02852
CPU times: user 70.6 ms, sys: 1.78 ms, total: 72.4 ms
Wall time: 72.1 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [7]:
%%time

log_model_none = LogisticRegression(random_state=21, fit_intercept=False, penalty=None)
crossval(10, X_train, y_train, log_model_none)

train - 0.66529   |   test - 0.62963
Average accuracy on crossval is 0.62537
train - 0.65705   |   test - 0.65926
Average accuracy on crossval is 0.62537
train - 0.66447   |   test - 0.57778
Average accuracy on crossval is 0.62537
train - 0.66529   |   test - 0.62963
Average accuracy on crossval is 0.62537
train - 0.66694   |   test - 0.62222
Average accuracy on crossval is 0.62537
train - 0.65952   |   test - 0.57778
Average accuracy on crossval is 0.62537
train - 0.65045   |   test - 0.69630
Average accuracy on crossval is 0.62537
train - 0.68673   |   test - 0.61481
Average accuracy on crossval is 0.62537
train - 0.66474   |   test - 0.62687
Average accuracy on crossval is 0.62537
train - 0.65651   |   test - 0.61940
Average accuracy on crossval is 0.62537
Std is 0.03302
CPU times: user 142 ms, sys: 2.36 ms, total: 145 ms
Wall time: 144 ms


In [8]:
%%time

log_model_l1 = OneVsRestClassifier(LogisticRegression(random_state=21, fit_intercept=False, penalty='l1', solver='liblinear'))
crossval(10, X_train, y_train, log_model_l1)

train - 0.61830   |   test - 0.54815
Average accuracy on crossval is 0.58903
train - 0.62737   |   test - 0.62222
Average accuracy on crossval is 0.58903
train - 0.60511   |   test - 0.54074
Average accuracy on crossval is 0.58903
train - 0.63644   |   test - 0.62222
Average accuracy on crossval is 0.58903
train - 0.62407   |   test - 0.55556
Average accuracy on crossval is 0.58903
train - 0.62325   |   test - 0.58519
Average accuracy on crossval is 0.58903
train - 0.61253   |   test - 0.63704
Average accuracy on crossval is 0.58903
train - 0.64716   |   test - 0.58519
Average accuracy on crossval is 0.58903
train - 0.63015   |   test - 0.59701
Average accuracy on crossval is 0.58903
train - 0.61367   |   test - 0.59701
Average accuracy on crossval is 0.58903
Std is 0.03129
CPU times: user 171 ms, sys: 2.27 ms, total: 174 ms
Wall time: 173 ms


In [9]:
%%time

log_model_el = LogisticRegression(random_state=21, fit_intercept=False, penalty='elasticnet', solver='saga')
crossval(10, X_train, y_train, log_model_el)

ValueError: 
All the 10 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/maksim/Library/Python/3.9/lib/python/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/maksim/Library/Python/3.9/lib/python/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Users/maksim/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py", line 1203, in fit
    raise ValueError("l1_ratio must be specified when penalty is elasticnet.")
ValueError: l1_ratio must be specified when penalty is elasticnet.


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [10]:
%%time

svm_model_base = SVC(probability=True, kernel='linear', random_state=21)
crossval(10, X_train, y_train, svm_model_base)

train - 0.70486   |   test - 0.65926
Average accuracy on crossval is 0.65871
train - 0.69662   |   test - 0.75556
Average accuracy on crossval is 0.65871
train - 0.69415   |   test - 0.62222
Average accuracy on crossval is 0.65871
train - 0.70239   |   test - 0.65185
Average accuracy on crossval is 0.65871
train - 0.69085   |   test - 0.65185
Average accuracy on crossval is 0.65871
train - 0.68920   |   test - 0.64444
Average accuracy on crossval is 0.65871
train - 0.69250   |   test - 0.72593
Average accuracy on crossval is 0.65871
train - 0.70074   |   test - 0.62222
Average accuracy on crossval is 0.65871
train - 0.69605   |   test - 0.61940
Average accuracy on crossval is 0.65871
train - 0.71087   |   test - 0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 1.36 s, sys: 8.26 ms, total: 1.36 s
Wall time: 1.37 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [11]:
%%time

svm_model_05 = SVC(probability=True, kernel='linear', random_state=21, C=0.5)
crossval(10, X_train, y_train, svm_model_05)

train - 0.66694   |   test - 0.63704
Average accuracy on crossval is 0.63349
train - 0.66612   |   test - 0.73333
Average accuracy on crossval is 0.63349
train - 0.67271   |   test - 0.60741
Average accuracy on crossval is 0.63349
train - 0.67354   |   test - 0.62963
Average accuracy on crossval is 0.63349
train - 0.67766   |   test - 0.64444
Average accuracy on crossval is 0.63349
train - 0.66529   |   test - 0.61481
Average accuracy on crossval is 0.63349
train - 0.66200   |   test - 0.68889
Average accuracy on crossval is 0.63349
train - 0.66529   |   test - 0.57037
Average accuracy on crossval is 0.63349
train - 0.67463   |   test - 0.59701
Average accuracy on crossval is 0.63349
train - 0.66804   |   test - 0.61194
Average accuracy on crossval is 0.63349
Std is 0.04471
CPU times: user 1.35 s, sys: 8.24 ms, total: 1.36 s
Wall time: 1.36 s


In [12]:
%%time

svm_model_2 = SVC(probability=True, kernel='linear', random_state=21, C=2)
crossval(10, X_train, y_train, svm_model_2)

train - 0.70734   |   test - 0.65926
Average accuracy on crossval is 0.66836
train - 0.71393   |   test - 0.75556
Average accuracy on crossval is 0.66836
train - 0.74526   |   test - 0.63704
Average accuracy on crossval is 0.66836
train - 0.71558   |   test - 0.66667
Average accuracy on crossval is 0.66836
train - 0.71146   |   test - 0.67407
Average accuracy on crossval is 0.66836
train - 0.70157   |   test - 0.63704
Average accuracy on crossval is 0.66836
train - 0.70651   |   test - 0.71852
Average accuracy on crossval is 0.66836
train - 0.70981   |   test - 0.64444
Average accuracy on crossval is 0.66836
train - 0.72405   |   test - 0.64925
Average accuracy on crossval is 0.66836
train - 0.72488   |   test - 0.64179
Average accuracy on crossval is 0.66836
Std is 0.03721
CPU times: user 1.42 s, sys: 10.7 ms, total: 1.43 s
Wall time: 1.45 s


In [13]:
%%time

svm_model_3 = SVC(probability=True, kernel='linear', random_state=21, C=3)
crossval(10, X_train, y_train, svm_model_3)

train - 0.71228   |   test - 0.63704
Average accuracy on crossval is 0.67949
train - 0.73042   |   test - 0.77778
Average accuracy on crossval is 0.67949
train - 0.76834   |   test - 0.65926
Average accuracy on crossval is 0.67949
train - 0.73124   |   test - 0.66667
Average accuracy on crossval is 0.67949
train - 0.71641   |   test - 0.70370
Average accuracy on crossval is 0.67949
train - 0.72712   |   test - 0.68889
Average accuracy on crossval is 0.67949
train - 0.71888   |   test - 0.71852
Average accuracy on crossval is 0.67949
train - 0.74361   |   test - 0.62963
Average accuracy on crossval is 0.67949
train - 0.74629   |   test - 0.66418
Average accuracy on crossval is 0.67949
train - 0.72570   |   test - 0.64925
Average accuracy on crossval is 0.67949
Std is 0.04227
CPU times: user 1.47 s, sys: 10.5 ms, total: 1.48 s
Wall time: 1.48 s


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [14]:
%%time

tree_model_base = DecisionTreeClassifier(max_depth=10, random_state=21)
crossval(10, X_train, y_train, tree_model_base)

train - 0.81039   |   test - 0.74074
Average accuracy on crossval is 0.72551
train - 0.77741   |   test - 0.74074
Average accuracy on crossval is 0.72551
train - 0.83347   |   test - 0.70370
Average accuracy on crossval is 0.72551
train - 0.79720   |   test - 0.76296
Average accuracy on crossval is 0.72551
train - 0.82440   |   test - 0.75556
Average accuracy on crossval is 0.72551
train - 0.80379   |   test - 0.68889
Average accuracy on crossval is 0.72551
train - 0.80709   |   test - 0.76296
Average accuracy on crossval is 0.72551
train - 0.80132   |   test - 0.65926
Average accuracy on crossval is 0.72551
train - 0.80807   |   test - 0.75373
Average accuracy on crossval is 0.72551
train - 0.80478   |   test - 0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562
CPU times: user 40.7 ms, sys: 1.62 ms, total: 42.3 ms
Wall time: 45.6 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [15]:
%%time

tree_model_15 = DecisionTreeClassifier(max_depth=15, random_state=21)
crossval(10, X_train, y_train, tree_model_15)

train - 0.95796   |   test - 0.82222
Average accuracy on crossval is 0.85459
train - 0.93075   |   test - 0.83704
Average accuracy on crossval is 0.85459
train - 0.95631   |   test - 0.83704
Average accuracy on crossval is 0.85459
train - 0.95301   |   test - 0.86667
Average accuracy on crossval is 0.85459
train - 0.95136   |   test - 0.88889
Average accuracy on crossval is 0.85459
train - 0.94724   |   test - 0.82222
Average accuracy on crossval is 0.85459
train - 0.95466   |   test - 0.90370
Average accuracy on crossval is 0.85459
train - 0.94971   |   test - 0.87407
Average accuracy on crossval is 0.85459
train - 0.95305   |   test - 0.83582
Average accuracy on crossval is 0.85459
train - 0.94316   |   test - 0.85821
Average accuracy on crossval is 0.85459
Std is 0.02682
CPU times: user 35.9 ms, sys: 1.11 ms, total: 37.1 ms
Wall time: 36.7 ms


In [16]:
%%time

tree_model_20 = DecisionTreeClassifier(max_depth=20, random_state=21)
crossval(10, X_train, y_train, tree_model_20)

train - 0.98846   |   test - 0.86667
Average accuracy on crossval is 0.88649
train - 0.99011   |   test - 0.91111
Average accuracy on crossval is 0.88649
train - 0.98681   |   test - 0.85926
Average accuracy on crossval is 0.88649
train - 0.98763   |   test - 0.91111
Average accuracy on crossval is 0.88649
train - 0.98928   |   test - 0.88148
Average accuracy on crossval is 0.88649
train - 0.98186   |   test - 0.85926
Average accuracy on crossval is 0.88649
train - 0.98846   |   test - 0.91852
Average accuracy on crossval is 0.88649
train - 0.99176   |   test - 0.89630
Average accuracy on crossval is 0.88649
train - 0.99094   |   test - 0.88060
Average accuracy on crossval is 0.88649
train - 0.98847   |   test - 0.88060
Average accuracy on crossval is 0.88649
Std is 0.02075
CPU times: user 37.2 ms, sys: 1.58 ms, total: 38.7 ms
Wall time: 37.9 ms


In [17]:
%%time

tree_model_25 = DecisionTreeClassifier(max_depth=25, random_state=21)
crossval(10, X_train, y_train, tree_model_25)

train - 1.00000   |   test - 0.85926
Average accuracy on crossval is 0.88649
train - 1.00000   |   test - 0.91852
Average accuracy on crossval is 0.88649
train - 0.99918   |   test - 0.86667
Average accuracy on crossval is 0.88649
train - 1.00000   |   test - 0.91111
Average accuracy on crossval is 0.88649
train - 0.99918   |   test - 0.88889
Average accuracy on crossval is 0.88649
train - 0.99835   |   test - 0.85185
Average accuracy on crossval is 0.88649
train - 0.99753   |   test - 0.92593
Average accuracy on crossval is 0.88649
train - 1.00000   |   test - 0.88148
Average accuracy on crossval is 0.88649
train - 1.00000   |   test - 0.88060
Average accuracy on crossval is 0.88649
train - 1.00000   |   test - 0.88060
Average accuracy on crossval is 0.88649
Std is 0.02371
CPU times: user 37.7 ms, sys: 1.3 ms, total: 39 ms
Wall time: 38.5 ms


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [18]:
%%time

rf_model_base = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
crossval(10, X_train, y_train, rf_model_base)

train - 0.96455   |   test - 0.88148
Average accuracy on crossval is 0.88722
train - 0.96208   |   test - 0.91852
Average accuracy on crossval is 0.88722
train - 0.96785   |   test - 0.86667
Average accuracy on crossval is 0.88722
train - 0.96455   |   test - 0.89630
Average accuracy on crossval is 0.88722
train - 0.96538   |   test - 0.91111
Average accuracy on crossval is 0.88722
train - 0.96538   |   test - 0.88148
Average accuracy on crossval is 0.88722
train - 0.97115   |   test - 0.91852
Average accuracy on crossval is 0.88722
train - 0.96867   |   test - 0.85185
Average accuracy on crossval is 0.88722
train - 0.97364   |   test - 0.88060
Average accuracy on crossval is 0.88722
train - 0.97941   |   test - 0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204
CPU times: user 390 ms, sys: 5.18 ms, total: 395 ms
Wall time: 395 ms


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [19]:
%%time

rf_model_1 = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=21)
crossval(10, X_train, y_train, rf_model_1)

train - 0.99505   |   test - 0.88889
Average accuracy on crossval is 0.90874
train - 0.99835   |   test - 0.94815
Average accuracy on crossval is 0.90874
train - 0.99670   |   test - 0.88148
Average accuracy on crossval is 0.90874
train - 0.99753   |   test - 0.93333
Average accuracy on crossval is 0.90874
train - 0.99505   |   test - 0.91852
Average accuracy on crossval is 0.90874
train - 0.99670   |   test - 0.88889
Average accuracy on crossval is 0.90874
train - 0.99670   |   test - 0.91852
Average accuracy on crossval is 0.90874
train - 0.99753   |   test - 0.91111
Average accuracy on crossval is 0.90874
train - 0.99753   |   test - 0.92537
Average accuracy on crossval is 0.90874
train - 0.99588   |   test - 0.87313
Average accuracy on crossval is 0.90874
Std is 0.02330
CPU times: user 416 ms, sys: 4.5 ms, total: 421 ms
Wall time: 421 ms


In [20]:
%%time

rf_model_2 = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=21)
crossval(10, X_train, y_train, rf_model_2)

train - 0.99918   |   test - 0.90370
Average accuracy on crossval is 0.91766
train - 0.99918   |   test - 0.96296
Average accuracy on crossval is 0.91766
train - 0.99918   |   test - 0.89630
Average accuracy on crossval is 0.91766
train - 1.00000   |   test - 0.94074
Average accuracy on crossval is 0.91766
train - 0.99918   |   test - 0.91852
Average accuracy on crossval is 0.91766
train - 0.99835   |   test - 0.89630
Average accuracy on crossval is 0.91766
train - 0.99918   |   test - 0.92593
Average accuracy on crossval is 0.91766
train - 0.99918   |   test - 0.89630
Average accuracy on crossval is 0.91766
train - 1.00000   |   test - 0.93284
Average accuracy on crossval is 0.91766
train - 1.00000   |   test - 0.90299
Average accuracy on crossval is 0.91766
Std is 0.02160
CPU times: user 840 ms, sys: 9.25 ms, total: 849 ms
Wall time: 849 ms


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [21]:
rf_model_2.fit(X_train, y_train)
predictions = rf_model_2.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f'Accuracy on test is {accuracy:.4f}')

Accuracy on test is 0.9290


In [22]:
df_analysis = pd.DataFrame({
    'test': y_test.values, 
    'pred': predictions
})

In [23]:
df_analysis[~(df_analysis['test'] == df_analysis['pred'])]['test'].value_counts().nlargest(1) / len(df_analysis[df_analysis['test'] == 0]) * 100

test
0    25.925926
Name: count, dtype: float64

Model makes the most errors for day 0 (Monday).
It gives wrong predictions in 26% cases when the true value is Monday.